In [ ]:
%pip install "transformers[torch]"
%pip install git+https://github.com/huggingface/trl.git

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)
from trl import SFTTrainer, SFTConfig

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B",
    torch_dtype="auto",    
    device_map="auto",  
    attn_implementation = "flash_attention_2"         
)
tokenizer   = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B", use_fast=True)

EOT_TOKEN = "\nResponse:"
dataset = load_dataset("GAIR/lima")
train_dataset = dataset["train"].shuffle(seed=42)

# 3. Define the formatting function
def format_lima_conversation(example):
    conversation = example['conversations']
    # Join turns with the EOT token. Add one at the very end.
    formatted_text = f"{EOT_TOKEN}".join(conversation) + tokenizer.eos_token
    return {"text": formatted_text}
    
# 4. Apply the formatting
train_dataset = train_dataset.map(format_lima_conversation, remove_columns=['conversations', 'source'])
training_args = SFTConfig(
    dataset_text_field="text",
    packing = True,
    max_length = 4096,
    per_device_train_batch_size = 1
    # remove_unused_columns= False
)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset = train_dataset
)

for i, batch in enumerate(trainer.get_train_dataloader()):
    if i == 0:
        print(batch)
        print(len(batch['position_ids']))
        print(batch['position_ids'][0])
        for j in batch['position_ids'][0]:
            if j == 0:
                print("seq found")

{'input_ids': tensor([[  4340,    311,    387,  ...,     13,    220, 151645]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0'), 'position_ids': tensor([[   0,    1,    2,  ..., 4093, 4094, 4095]], device='cuda:0'), 'labels': tensor([[  4340,    311,    387,  ...,     13,    220, 151645]],
       device='cuda:0')}
1
tensor([   0,    1,    2,  ..., 4093, 4094, 4095], device='cuda:0')
seq found


In [ ]:

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)

model = AutoModelForCausalLM.from_pretrained(
    "jiosephlee/olmo2-lima",
    torch_dtype="auto",    
    device_map="auto",  
    attn_implementation = "flash_attention_2"         
)
tokenizer   = AutoTokenizer.from_pretrained("jiosephlee/olmo2-lima", use_fast=True)

EOT_TOKEN = "\nResponse:"

def generate_text(model, tokenizer, prompt: str) -> str:
    """Simple inference function using Hugging Face transformers.generate."""
    inputs = tokenizer(prompt , return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=False)